In [3]:
import carla
import logging
import time 
import cv2
import random

In [18]:
CARLA_LOCALHOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 20.0

logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)

logger = logging.getLogger(__name__)
CAMERA_POS_Z = 3
CAMERA_POS_X = 0
CAMERA_INIT_TRANS = carla.Transform(carla.Location(z = CAMERA_POS_Z, x = CAMERA_POS_X))

In [15]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(CARLA_LOCALHOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT)
    world = client.get_world()

    if not world:
        raise RuntimeError(
            "Carla was not able to return a valid world."
        )


    logger.info("## Client conection established. ##")

    return client, world

In [6]:
def spawn_point(world: carla.World) -> carla.Transform:

    spawn_points = world.get_map().get_spawn_points()

    if not spawn_points:
        raise RuntimeError(
            "No spawn point retrieved from the map"
        )

    logger.info("## Ego vehicle spawn point selected.## ")
    return spawn_points[0]


In [17]:
def setup_environment(world: carla.World, client: carla.Client, map: str = 'Town01') -> None:

    try:
        world = client.load_world(map)
    except:
        print("\nIt was not able to load selected town. It defaults to Town01.\n")
    finally:
        logger.info(f"## Map {map} is loaded. ##")

    


In [9]:
def ego_vehicle(world: carla.World, spawn_point: carla.Transform) -> carla.Vehicle:

    vehicle_bp = world.get_blueprint_library().filter("vehicle.*")

    vehicle = world.spawn_actor(
        vehicle_bp[0],
        spawn_point
    )

    logger.info("## Ego vehicle loaded ##")

    return vehicle

In [22]:
def spawn_gnss_sensor(world: carla.World, vehicle: carla.Vehicle) -> carla.Sensor:

    gnss_bp = world.get_blueprint_library().find("sensor.other.gnss")

    if not gnss_bp:
        raise RuntimeError(
            "No valide BP for GNSS was found."
        )

    gnss = world.spawn_actor(gnss_bp, CAMERA_INIT_TRANS, attach_to=vehicle)
    logger.info("## GNSS Sensor attached to the vehicle. ##")

    return gnss

In [32]:
def gnss_callback(data, data_dict):
    data_dict['gnss'] = [data.latitude, data.longitude]

In [40]:
def main():

    client, world = connect_to_carla()
    spawn_location = spawn_point(world)
    actors = world.get_actors().filter("vehicle.*")
    if actors is not None:
        for vehicle in actors:
            vehicle.destroy()
    #setup_environment(world, client, 'Town02')
    vehicle = ego_vehicle(world, spawn_location)
    gnss = spawn_gnss_sensor(world, vehicle)
    vehicle.set_autopilot(True)
    

    data_dict = {
        "gnss": None
    }

    gnss.listen(
        lambda data: gnss_callback(
            data,
            data_dict
        )
    )
    try:
        while True:
            gnss_data = data_dict.get("gnss")
            print(gnss_data)
            time.sleep(0.1)
    except KeyboardInterrupt:
        print("\n Stopping GNSS..")

    finally:
        gnss.stop()
        gnss.destroy()
        vehicle.destroy()


In [ ]:
if __name__ == "__main__":
    main() 

2026-11-11 21:48:27,009 [INFO] __main__: ## Client conection established. ##
2026-11-11 21:48:27,079 [INFO] __main__: ## Ego vehicle spawn point selected.## 
2026-11-11 21:48:27,113 [INFO] __main__: ## Ego vehicle loaded ##
2026-11-11 21:48:27,132 [INFO] __main__: ## GNSS Sensor attached to the vehicle. ##


None
[0.0012778256546820549, -6.767020659488414e-05]
[0.0012778023431115923, -6.767020659488353e-05]
[0.0012777950753866834, -6.767018516880753e-05]
[0.0012777953496404535, -6.767017659837723e-05]
[0.0012777968580361893, -6.767017659837726e-05]
[0.001277797817924385, -6.767018516880761e-05]
[0.0012777983664319253, -6.767018516880761e-05]
[0.0012780384756076897, -6.767081081022725e-05]
[0.001279212693124578, -6.766304600038896e-05]
[0.0012809787502774455, -6.764712642611751e-05]
[0.0012829311628671286, -6.762870000098266e-05]
[0.0012852406538655442, -6.760665685426258e-05]
[0.001287664920066767, -6.758330243170675e-05]
[0.0012903789353761505, -6.755679409080119e-05]
[0.0012932915104151219, -6.752784746247519e-05]
[0.0012965370295310527, -6.749507413702058e-05]
[0.0012998283490265976, -6.746136234944735e-05]
[0.0013037115081581225, -6.742105990097454e-05]
[0.0013078279201211578, -6.737777065754187e-05]
[0.0013121832071179922, -6.733164460168013e-05]
[0.0013166789120451425, -6.72836887589

: 